# Lab 3 — Model Comparison

## Objective
The goal of this lab is to compare multiple models for predicting how many championship points a driver will score in a specific Formula 1 Grand Prix.

This problem is framed as a regression task, where the target variable is `points`. The primary evaluation metric is Mean Absolute Error (MAE), since it measures prediction error directly in championship points.

The comparison will use the same dataset, the same temporal validation strategy, and the same metric across all models.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

RANDOM_SEED = 414
np.random.seed(RANDOM_SEED)

## 1. Data Loading
We load the race results dataset and inspect its structure before defining the modeling pipeline.

In [52]:
df = pd.read_csv("jolpica_2019_2024_results.csv")
df.head()

,season,round,race,driver,constructor,grid,position_str,points
0,2019,1,Australian Grand Prix,Charles Leclerc,Ferrari,4,1,25.0
1,2019,1,Australian Grand Prix,Lando Norris,McLaren,12,2,18.0
2,2019,1,Australian Grand Prix,Max Verstappen,Red Bull,2,3,15.0
3,2019,1,Australian Grand Prix,Daniel Ricciardo,Renault,18,4,12.0
4,2019,1,Australian Grand Prix,Kevin Magnussen,Haas,13,R,0.0


In [53]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (2340, 8)

Columns:
['season', 'round', 'race', 'driver', 'constructor', 'grid', 'position_str', 'points']


In [54]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2340 entries, 0 to 2339
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   season        2340 non-null   int64  
 1   round         2340 non-null   int64  
 2   race          2340 non-null   str    
 3   driver        2340 non-null   str    
 4   constructor   2340 non-null   str    
 5   grid          2340 non-null   int64  
 6   position_str  2340 non-null   str    
 7   points        2340 non-null   float64
dtypes: float64(1), int64(3), str(4)
memory usage: 146.4 KB


In [55]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
season,2340.0,NaN,NaN,NaN,2021.521368,1.727965,2019.0,2020.0,2022.0,2023.0,2024.0
round,2340.0,NaN,NaN,NaN,10.290598,5.689801,1.0,5.0,10.0,15.0,21.0
race,2340,33,Bahrain Grand Prix,120,NaN,NaN,NaN,NaN,NaN,NaN,NaN
driver,2340,29,Charles Leclerc,117,NaN,NaN,NaN,NaN,NaN,NaN,NaN
constructor,2340,13,Ferrari,234,NaN,NaN,NaN,NaN,NaN,NaN,NaN
grid,2340.0,NaN,NaN,NaN,10.594444,5.764664,1.0,5.0,11.0,16.0,20.0
position_str,2340,21,R,236,NaN,NaN,NaN,NaN,NaN,NaN,NaN
points,2340.0,NaN,NaN,NaN,4.526496,7.002637,0.0,0.0,0.0,8.0,25.0


## 2. Target and Candidate Features

The target variable is `points`, which represents the championship points scored by a driver in a race.

Initial candidate predictors include:
- `grid`: starting position
- `driverRef`: driver identity
- `constructorRef`: team identity
- `season`: year of the race

These variables are available before or at race start, or summarize stable race context.

In [56]:
selected_cols = ["season", "grid", "points", "driver", "constructor"]
data = df[selected_cols].copy()
data.head()

,season,grid,points,driver,constructor
0,2019,4,25.0,Charles Leclerc,Ferrari
1,2019,12,18.0,Lando Norris,McLaren
2,2019,2,15.0,Max Verstappen,Red Bull
3,2019,18,12.0,Daniel Ricciardo,Renault
4,2019,13,0.0,Kevin Magnussen,Haas


In [57]:
df.columns.tolist()

['season',
 'round',
 'race',
 'driver',
 'constructor',
 'grid',
 'position_str',
 'points']

In [58]:
data = data.dropna(subset=["season", "grid", "points", "driver", "constructor"]).copy()

data["season"] = data["season"].astype(int)
data["grid"] = data["grid"].astype(int)
data["points"] = data["points"].astype(float)

print(data.shape)
data.head()

(2340, 5)


,season,grid,points,driver,constructor
0,2019,4,25.0,Charles Leclerc,Ferrari
1,2019,12,18.0,Lando Norris,McLaren
2,2019,2,15.0,Max Verstappen,Red Bull
3,2019,18,12.0,Daniel Ricciardo,Renault
4,2019,13,0.0,Kevin Magnussen,Haas


## 3. Exploratory Analysis

Before training models, we inspect the target distribution. In Formula 1, many race entries score zero points, which makes this regression problem difficult and slightly skewed.

In [59]:
print(data["points"].describe())
print("\nShare of zero-point results:", (data["points"] == 0).mean())

count    2340.000000
mean        4.526496
std         7.002637
min         0.000000
25%         0.000000
50%         0.000000
75%         8.000000
max        25.000000
Name: points, dtype: float64

Share of zero-point results: 0.5508547008547009


In [60]:
data["points"].value_counts().sort_index().head(20)

points
0.0     1289
1.0      110
2.0      108
4.0      108
6.0      102
8.0      107
10.0      96
12.0      98
15.0     105
18.0     110
25.0     107
Name: count, dtype: int64

## 4. Temporal Validation Strategy

A random split is not appropriate for this task because the lab requires temporal validation.  
To simulate a realistic forecasting setting, the models are trained on past seasons and evaluated on a future season.

- Train: 2019–2023
- Test: 2024

In [61]:
train_df = data[data["season"] < 2024].copy()
test_df = data[data["season"] == 2024].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train seasons:", sorted(train_df["season"].unique()))
print("Test seasons:", sorted(test_df["season"].unique()))

Train shape: (1940, 5)
Test shape: (400, 5)
Train seasons: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Test seasons: [np.int64(2024)]


## 5. Feature Engineering

We use a mix of numeric and categorical features:
- `grid` as a numeric predictor
- driver and constructor as categorical predictors

To keep the pipeline simple and reproducible, categorical variables are encoded as integer category codes.

In [62]:
all_drivers = pd.Categorical(pd.concat([train_df["driver"], test_df["driver"]]))
all_constructors = pd.Categorical(pd.concat([train_df["constructor"], test_df["constructor"]]))

driver_map = {cat: i for i, cat in enumerate(all_drivers.categories)}
constructor_map = {cat: i for i, cat in enumerate(all_constructors.categories)}

train_df["driver_cat"] = train_df["driver"].map(driver_map)
test_df["driver_cat"] = test_df["driver"].map(driver_map)

train_df["constructor_cat"] = train_df["constructor"].map(constructor_map)
test_df["constructor_cat"] = test_df["constructor"].map(constructor_map)

feature_cols = ["grid", "driver_cat", "constructor_cat"]

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
y_train = train_df["points"]
y_test = test_df["points"]

X_train.head()

,grid,driver_cat,constructor_cat
0,4,3,4
1,12,12,6
2,2,15,9
3,18,4,10
4,13,9,5


## 6. Baseline 1 — Global Mean

This baseline predicts the same value for every case: the average number of points observed in the training set.

It is intentionally simple and provides a minimum standard that any useful model should beat.

In [63]:
global_mean = y_train.mean()

baseline1_train_pred = np.full(len(y_train), global_mean)
baseline1_test_pred = np.full(len(y_test), global_mean)

baseline1_train_mae = mean_absolute_error(y_train, baseline1_train_pred)
baseline1_test_mae = mean_absolute_error(y_test, baseline1_test_pred)

print("Baseline 1 - Train MAE:", baseline1_train_mae)
print("Baseline 1 - Test MAE:", baseline1_test_mae)

Baseline 1 - Train MAE: 5.612902540121161
Baseline 1 - Test MAE: 5.552177835051548


## 7. Baseline 2 — Grid Heuristic

This domain baseline predicts points based on the historical average points associated with each starting grid position in the training data.

This is stronger than the global mean because starting position is strongly related to race outcomes in Formula 1.

In [64]:
grid_avg_points = train_df.groupby("grid")["points"].mean()

baseline2_train_pred = train_df["grid"].map(grid_avg_points).fillna(global_mean)
baseline2_test_pred = test_df["grid"].map(grid_avg_points).fillna(global_mean)

baseline2_train_mae = mean_absolute_error(y_train, baseline2_train_pred)
baseline2_test_mae = mean_absolute_error(y_test, baseline2_test_pred)

print("Baseline 2 - Train MAE:", baseline2_train_mae)
print("Baseline 2 - Test MAE:", baseline2_test_mae)

Baseline 2 - Train MAE: 5.561859237494068
Baseline 2 - Test MAE: 5.608548802912555


## 8. Model 1 — Linear Regression

Linear Regression is used as a simple interpretable model.  
It assumes additive and linear relationships between the predictors and the target.

In [65]:
lr = LinearRegression()
lr.fit(X_train, y_train)

lr_train_pred = lr.predict(X_train)
lr_test_pred = lr.predict(X_test)

lr_train_mae = mean_absolute_error(y_train, lr_train_pred)
lr_test_mae = mean_absolute_error(y_test, lr_test_pred)

print("Linear Regression - Train MAE:", lr_train_mae)
print("Linear Regression - Test MAE:", lr_test_mae)

Linear Regression - Train MAE: 5.598763341796171
Linear Regression - Test MAE: 5.548003059323432


## 9. Model 2 — Random Forest

Random Forest can capture nonlinear relationships and interactions between variables.  
This is useful in F1 because the effect of grid position and team strength is unlikely to be purely linear.

In [66]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=3,
    random_state=RANDOM_SEED
)

rf.fit(X_train, y_train)

rf_train_pred = rf.predict(X_train)
rf_test_pred = rf.predict(X_test)

rf_train_mae = mean_absolute_error(y_train, rf_train_pred)
rf_test_mae = mean_absolute_error(y_test, rf_test_pred)

print("Random Forest - Train MAE:", rf_train_mae)
print("Random Forest - Test MAE:", rf_test_mae)

Random Forest - Train MAE: 4.926595613117626
Random Forest - Test MAE: 5.860231056638031


## 10. Model 3 — XGBoost

XGBoost is a stronger tree-based model for tabular data.  
It often performs well when the data contains nonlinear patterns and interactions.

In [67]:
xgb = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED,
    objective="reg:squarederror"
)

xgb.fit(X_train, y_train)

xgb_train_pred = xgb.predict(X_train)
xgb_test_pred = xgb.predict(X_test)

xgb_train_mae = mean_absolute_error(y_train, xgb_train_pred)
xgb_test_mae = mean_absolute_error(y_test, xgb_test_pred)

print("XGBoost - Train MAE:", xgb_train_mae)
print("XGBoost - Test MAE:", xgb_test_mae)

XGBoost - Train MAE: 5.2431295730480825
XGBoost - Test MAE: 5.707563648447394


## 11. Model Comparison Table

All rows below use:
- the same target (`points`)
- the same metric (MAE)
- the same temporal split (train: 2019–2023, test: 2024)

This makes the comparison valid and fair.

In [68]:
comparison = pd.DataFrame([
    {
        "Model": "Baseline 1 - Global Mean",
        "Train MAE": baseline1_train_mae,
        "Test MAE": baseline1_test_mae,
        "Gap": baseline1_test_mae - baseline1_train_mae,
        "WHY": "Ignores all race context and predicts the same value every time."
    },
    {
        "Model": "Baseline 2 - Grid Heuristic",
        "Train MAE": baseline2_train_mae,
        "Test MAE": baseline2_test_mae,
        "Gap": baseline2_test_mae - baseline2_train_mae,
        "WHY": "Uses starting position, which is one of the strongest simple predictors of points."
    },
    {
        "Model": "Linear Regression",
        "Train MAE": lr_train_mae,
        "Test MAE": lr_test_mae,
        "Gap": lr_test_mae - lr_train_mae,
        "WHY": "Captures broad linear trends but may miss nonlinear effects between grid, driver, and constructor."
    },
    {
        "Model": "Random Forest",
        "Train MAE": rf_train_mae,
        "Test MAE": rf_test_mae,
        "Gap": rf_test_mae - rf_train_mae,
        "WHY": "Captures nonlinear patterns and interactions, but may overfit if the train-test gap becomes large."
    },
    {
        "Model": "XGBoost",
        "Train MAE": xgb_train_mae,
        "Test MAE": xgb_test_mae,
        "Gap": xgb_test_mae - xgb_train_mae,
        "WHY": "Can model more complex tabular relationships and often generalizes better than a single tree ensemble."
    }
])

comparison = comparison.sort_values("Test MAE").reset_index(drop=True)
comparison

,Model,Train MAE,Test MAE,Gap,WHY
0,Linear Regression,5.598763,5.548003,-0.050760,Captures broad linear trends but may miss nonl...
1,Baseline 1 - Global Mean,5.612903,5.552178,-0.060725,Ignores all race context and predicts the same...
2,Baseline 2 - Grid Heuristic,5.561859,5.608549,0.046690,"Uses starting position, which is one of the st..."
3,XGBoost,5.243130,5.707564,0.464434,Can model more complex tabular relationships a...
4,Random Forest,4.926596,5.860231,0.933635,"Captures nonlinear patterns and interactions, ..."


## 12. Interpretation

The best-performing model is **Linear Regression**, with the lowest Test MAE (~5.55), slightly outperforming both baselines and more complex models.

### Baselines
The **Global Mean baseline** achieved a Test MAE of ~5.55, which is very close to Linear Regression. This indicates that predicting the average number of points is already a strong benchmark, mainly because a large proportion of drivers score zero points.

The **Grid Heuristic baseline** performed slightly worse (~5.61 MAE), but still competitive. This confirms that starting position is an important predictor in Formula 1, although not sufficient on its own to fully explain race outcomes.

### Linear Regression
Linear Regression achieved the best Test MAE (~5.55) and shows almost no gap between train and test performance. This suggests that the model is stable and not overfitting.

However, its improvement over the Global Mean baseline is minimal. This indicates that, while grid, driver, and constructor provide useful information, their linear combination does not significantly improve predictive power in this setup.

### Random Forest
Random Forest achieved a much lower Train MAE (~4.93) but a significantly worse Test MAE (~5.86). The large train-test gap indicates clear overfitting.

This suggests that the model is capturing noise or overly specific patterns from past seasons that do not generalize well to new races. Given the relatively small dataset (~2000 rows), this behavior is expected for a flexible model like Random Forest.

### XGBoost
XGBoost performed better than Random Forest (Test MAE ~5.71) but still worse than both Linear Regression and the baselines. It also shows a noticeable train-test gap, though smaller than Random Forest.

This suggests that while XGBoost can model more complex relationships, the available features are not rich enough to support that complexity. As a result, the model cannot translate its flexibility into better generalization.

### Key Insight
Overall, none of the models significantly outperform the baselines. This indicates that the prediction task is inherently difficult with the current feature set.

A major reason is the high proportion of zero-point outcomes (over 50% of observations), which makes the problem highly skewed and reduces the ability of models to distinguish meaningful patterns.

Additionally, important factors such as race incidents, weather conditions, safety cars, and strategy decisions are not included in the dataset, limiting the predictive power of all models.

## 13. Conclusion

This lab compared two baselines and three regression models for predicting Formula 1 championship points at the race level.

The results show that **Linear Regression performs best**, but only marginally better than a simple Global Mean baseline. This suggests that the current features (grid position, driver, and constructor) capture some structure in the data, but not enough to produce strong predictive improvements.

More complex models such as Random Forest and XGBoost did not outperform simpler approaches and showed signs of overfitting, indicating that increased model complexity is not beneficial given the current dataset.

From a practical perspective, this means that while it is possible to estimate expected points at a coarse level, predictions remain uncertain and should be used cautiously for strategic decisions.

The main limitation of this work is the lack of key race context variables, such as weather, incidents, and race strategy, which play a crucial role in determining final outcomes in Formula 1.